In [2]:
# Nhìn lại dữ liệu M3 theo đơn vị
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path("..") / "data" / "raw" / "evtx_data.csv", low_memory=False)
print(df.shape)
print(df["EVTX_FileName"].nunique(),"file | ", df["EVTX_Tactic"].nunique(),"tactic")
events_per_file = df["EVTX_FileName"].value_counts()
print("event/file: trung vị", int(events_per_file.median()),
      "| lớn nhất", events_per_file.max())
print(df.groupby("EVTX_Tactic")["EVTX_FileName"].nunique().sort_values().to_dict())


(4633, 326)
249 file |  8 tactic
event/file: trung vị 5 | lớn nhất 869
{'Command and Control': 5, 'Discovery': 10, 'Persistence': 22, 'Execution': 33, 'Credential Access': 35, 'Defense Evasion': 36, 'Lateral Movement': 47, 'Privilege Escalation': 61}


In [3]:
#  Hàm normalize
import re
 
def normalize(text):
    t = str(text).lower()
    t = re.sub(r"\busers([\\/])[^\\/\s\"']+", r"users\1<user>", t)   # tên người dùng
    t = re.sub(r"\{?[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\}?",
               " <guid> ", t)                                        # GUID
    t = re.sub(r"\b\d{1,3}(?:\.\d{1,3}){3}\b", " <ip> ", t)            # địa chỉ IPv4
    return t
 
tests = [r"C:\Users\IEUser\Desktop\mimikatz.exe",
         r"\\10.0.2.17\C$\users\ieuser\AppData\x.dll",
         r"{4A8F9C2E-1B3D-4E5F-9A0B-1C2D3E4F5A6B} C:/Users/den/a.ps1"]
for s in tests:
    print(normalize(s))

c:\users\<user>\desktop\mimikatz.exe
\\ <ip> \c$\users\<user>\appdata\x.dll
 <guid>  c:/users/<user>/a.ps1


In [4]:
#  Hàm build_doc và các phép kiểm tra
BEHAVIOR = ["Image", "ParentImage", "CommandLine", "NewProcessName", "ProcessName",
            "Application", "ShareName", "RelativeTargetName", "TargetFilename",
            "ImageLoaded", "TargetImage", "GrantedAccess", "PipeName", "TargetObject",
            "LogonType", "EventType", "TaskName", "AccessMask"]
 
def build_doc(row):
    # tên kênh ngắn gọn
    channel = str(row["Channel"]).split("/")[0].split("-")[-1].lower()
    parts = [f"ch_{channel}", f"eid_{row['EventID']}"]                  # 2 token cố định
    for col in BEHAVIOR:
        val = row[col]
        # bỏ ô trống và "-"
        if pd.notna(val) and str(val) not in ("-", ""):
            parts.append(normalize(val))
    return " ".join(parts)
 
docs = df.apply(build_doc, axis=1)                    # áp lên TỪNG DÒNG
y = df["EVTX_Tactic"]
groups = df["EVTX_FileName"]                          # chỉ để CHIA, không làm feature
print(docs.iloc[0])
print(docs.str.split().str.len().describe()[["mean", "50%", "max"]].round(1).to_dict())
 
assert not docs.str.contains("ieuser|user01").any(), "còn sót tên người dùng"
stems = groups.str.lower().str.replace(".evtx", "", regex=False).unique()
# tên file lọt vào docs?
leak = [s for s in stems if docs.str.contains(s, regex=False).any()]
print("tên file xuất hiện trong docs:", leak)

ch_security eid_4624 c:\windows\system32\services.exe 5.0
{'mean': 4.6, '50%': 4.0, 'max': 43.0}
tên file xuất hiện trong docs: ['roguewinrm']


In [5]:
# TF-IDF trên tài liệu M3
from sklearn.feature_extraction.text import TfidfVectorizer
 
TOKEN = r"(?u)[\w\.\-<>]+"
line = r"C:\Windows\System32\cmd.exe /c whoami -enc <user>"
print(TfidfVectorizer().build_analyzer()(line))
print(TfidfVectorizer(token_pattern=TOKEN).build_analyzer()(line))
 
v = TfidfVectorizer(token_pattern=TOKEN, min_df=2).fit(docs)
print("vocabulary:", len(v.vocabulary_))
idf = pd.Series(v.idf_, index=v.get_feature_names_out())
check = ["cmd.exe", "powershell.exe", "sdbinst.exe", "<user>", "eid_4624"]
print(idf[check].round(2).to_dict())

['windows', 'system32', 'cmd', 'exe', 'whoami', 'enc', 'user']
['c', 'windows', 'system32', 'cmd.exe', 'c', 'whoami', '-enc', '<user>']
vocabulary: 1278
{'cmd.exe': 3.84, 'powershell.exe': 4.69, 'sdbinst.exe': 5.57, '<user>': 2.29, 'eid_4624': 5.19}


In [6]:
# Đóng gói thành src/features.py
"""Dựng "tài liệu" văn bản từ event EVTX cho TF-IDF (dùng chung cho cấu hình A, B, C)."""
import re
from pathlib import Path
 
import pandas as pd
 
BEHAVIOR = ["Image", "ParentImage", "CommandLine", "NewProcessName", "ProcessName",
            "Application", "ShareName", "RelativeTargetName", "TargetFilename",
            "ImageLoaded", "TargetImage", "GrantedAccess", "PipeName", "TargetObject",
            "LogonType", "EventType", "TaskName", "AccessMask"]
TOKEN = r"(?u)[\w\.\-<>]+"   # giữ dấu chấm, gạch ngang và nhãn <user>, <guid>, <ip>
 
 
def normalize(text):
    """Viết thường và thay định danh (người dùng, GUID, IP) bằng nhãn chung."""
    t = str(text).lower()
    t = re.sub(r"\busers([\\/])[^\\/\s\"']+", r"users\1<user>", t)
    t = re.sub(r"\{?[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\}?",
               " <guid> ", t)
    t = re.sub(r"\b\d{1,3}(?:\.\d{1,3}){3}\b", " <ip> ", t)
    return t
 
 
def build_doc(row):
    """Một event -> một chuỗi: kênh, EventID và các trường hành vi đã chuẩn hóa."""
    channel = str(row["Channel"]).split("/")[0].split("-")[-1].lower()
    parts = [f"ch_{channel}", f"eid_{row['EventID']}"]
    for col in BEHAVIOR:
        val = row[col]
        if pd.notna(val) and str(val) not in ("-", ""):
            parts.append(normalize(val))
    return " ".join(parts)
 
 
def load_m3(path):
    """Đọc evtx_data.csv, trả về (df, docs, y, groups)."""
    df = pd.read_csv(Path(path), low_memory=False)
    docs = df.apply(build_doc, axis=1)
    assert not docs.str.contains("ieuser|user01").any(), "còn sót tên người dùng"
    return df, docs, df["EVTX_Tactic"], df["EVTX_FileName"]

In [7]:
import sys
sys.path.append(str(Path("..").resolve()))           # để Python thấy thư mục src/
 
import importlib, src.features
importlib.reload(src.features)
from src.features import load_m3, TOKEN
 
df2, docs2, y2, groups2 = load_m3("../data/raw/evtx_data.csv")
print(len(docs2), (docs2 == docs).all())               # giống hệt bản trong notebook

4633 True
